# Milking data: cleaning and preprocessing
Full-data audit: 8,495,421 rows, 11 columns, 9,087 known AnimalId values.
Preserve the original CSV. Audit and flag before applying analysis-specific exclusions.
Do not impute IDs, drop every incomplete row, or automatically remove all IQR outliers.
Exact duplicates are retained by default. Set DROP_EXACT_DUPLICATES=True only after confirming repeated exports.
Requires pandas, numpy and matplotlib; full execution needs several GB of memory.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path(r"D:\ansci-4040-fall-2026\jl4937_6040-project-1")
DATA_PATH = ROOT / "dataset" / "Data_set_prep_assignment_1.csv"
OUT = ROOT / "outputs" / "data_cleaning"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, dtype={"AnimalId": "string", "ReproductionStatus": "string"}, low_memory=False)
original_columns = df.columns.tolist()
raw_rows = len(df)
summary = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing_n": df.isna().sum(),
                        "missing_pct": df.isna().mean().mul(100), "unique_n": df.nunique()})
display(summary)
display(df.describe(percentiles=[.01, .5, .99]).T)
summary.to_csv(OUT / "raw_column_audit.csv", encoding="utf-8-sig")


## 1. Duplicates and types
Read AnimalId as string from the start to protect integer precision. Negative IDs are not invalid.
The candidate event key is AnimalId + LactationNumber + EventDate + milking. Its uniqueness is not established.
Review conflicting records instead of automatically averaging or discarding them.
Duplicate comparisons must exclude the added source row identifier.

In [ ]:
DROP_EXACT_DUPLICATES = False
exact_duplicate = df.duplicated(subset=original_columns, keep="first")
print("Exact duplicate excess:", int(exact_duplicate.sum()))

df["source_csv_line"] = np.arange(len(df), dtype=np.int64) + 2
if DROP_EXACT_DUPLICATES:
    df = df.loc[~exact_duplicate].copy()
df["flag_exact_duplicate"] = exact_duplicate.loc[df.index]
date_before = df["EventDate"].copy()
df["EventDate"] = pd.to_datetime(date_before, format="%Y-%m-%d", errors="coerce")
df["flag_invalid_date"] = date_before.notna() & df["EventDate"].isna()
for c in ["AnimalId", "ReproductionStatus"]:
    df[c] = df[c].str.strip().replace("", pd.NA)

key = ["AnimalId", "LactationNumber", "EventDate", "milking"]
complete_key = df[key].notna().all(axis=1)
df["flag_key_conflict"] = complete_key & df.duplicated(key, keep=False)

unique_records = df.loc[~df["flag_exact_duplicate"] & complete_key]
print("Candidate-key duplicate excess after exact dedup:", int(unique_records.duplicated(key).sum()))
display(unique_records.loc[unique_records.duplicated(key, keep=False), original_columns].head(20))
del unique_records, date_before, exact_duplicate


## 2. Missingness and cross-field consistency
Records without identity may support event-level summaries but cannot support reliable animal grouping.
First-two-minute yield exceeding total yield is inconsistent if both are cumulative measurements of the same event in the same units. Confirm definitions.
Zero average flow with positive yield needs review. Zero flow during seconds 30-60 may be real.
Do not invent upper limits for lactation number, days in milk or milk flow.
The flow ratio is diagnostic only; do not automatically use the derived flow to replace measured flow.

In [ ]:
identity = ["AnimalId", "LactationNumber", "DaysInMilk", "ReproductionStatus"]
df["flag_missing_identity"] = df[identity].isna().any(axis=1)
df["flag_first2_gt_total"] = df["YieldFirst2Min_Session"] > df["YieldSession"]
df["flag_zero_avg_positive_yield"] = df["Avgmilkflow"].eq(0) & df["YieldSession"].gt(0)
nonnegative = ["DaysInMilk", "Avgmilkflow", "Flow30_60Session", "YieldFirst2Min_Session", "YieldSession"]
df["flag_negative_measurement"] = df[nonnegative].lt(0).any(axis=1)
df["flag_bad_duration"] = df["DurationSession_sec"].le(0) | df["DurationSession_sec"].isna()
df["flag_bad_lactation"] = df["LactationNumber"].notna() & (df["LactationNumber"].lt(1) | df["LactationNumber"].mod(1).ne(0))
df["flag_bad_dim"] = df["DaysInMilk"].notna() & df["DaysInMilk"].mod(1).ne(0)
df["flag_bad_milking"] = ~df["milking"].isin([1, 2, 3])

derived_flow = df["YieldSession"] / (df["DurationSession_sec"].where(df["DurationSession_sec"].gt(0)) / 60)
df["flow_ratio_diagnostic"] = df["Avgmilkflow"] / derived_flow.where(derived_flow.gt(0))
flag_cols = [c for c in df if c.startswith("flag_")]
flag_summary = df[flag_cols].sum().sort_values(ascending=False).to_frame("n")
flag_summary["pct"] = flag_summary["n"] / len(df) * 100
display(flag_summary)
flag_summary.to_csv(OUT / "flag_summary.csv", encoding="utf-8-sig")

display(df.groupby("flag_missing_identity")[["YieldSession", "DurationSession_sec", "Avgmilkflow"]].agg(["count", "mean", "median"]))
monthly_missing = df.groupby(df["EventDate"].dt.to_period("M"))["flag_missing_identity"].agg(["size", "mean"])
display(monthly_missing)


## 3. Longitudinal consistency and distribution checks
Within each animal and lactation, EventDate minus DaysInMilk should approximately identify the same lactation start date.
Review groups with large spreads; small differences may reflect recording conventions.
Plots use a reproducible sample. Missingness and rule counts above use all records.

In [ ]:
track = df.loc[df[["AnimalId", "LactationNumber", "DaysInMilk", "EventDate"]].notna().all(axis=1),
               ["AnimalId", "LactationNumber", "DaysInMilk", "EventDate"]].copy()
track["implied_start"] = track["EventDate"] - pd.to_timedelta(track["DaysInMilk"], unit="D")
start_audit = track.groupby(["AnimalId", "LactationNumber"])["implied_start"].agg(["min", "max", "size"])
start_audit["spread_days"] = (start_audit["max"] - start_audit["min"]).dt.days
display(start_audit.sort_values("spread_days", ascending=False).head(20))
start_audit.to_csv(OUT / "lactation_date_audit.csv", encoding="utf-8-sig")
del track
import matplotlib.pyplot as plt
sample = df.sample(n=min(100000, len(df)), random_state=42)
sample[["DaysInMilk", "Avgmilkflow", "YieldSession", "DurationSession_sec"]].hist(bins=60, figsize=(12, 7))
plt.tight_layout()
plt.show()


## 4. Analysis masks and audit exports
The event mask excludes invalid dates, negative measurements, nonpositive durations and invalid discrete fields.
The animal mask additionally requires complete identity metadata. Other flags remain for review and sensitivity analysis.
These are starting masks, not a universal final cleaned cohort. No automatic imputation or winsorization is performed.
Only small audit files are exported by default. EXPORT_FULL=True exports all retained records with flags.

In [ ]:
hard_flags = ["flag_invalid_date", "flag_negative_measurement", "flag_bad_duration", "flag_bad_milking", "flag_bad_lactation", "flag_bad_dim"]
event_mask = ~df[hard_flags].any(axis=1) & df["EventDate"].notna()
animal_mask = event_mask & ~df["flag_missing_identity"]
review_mask = df[flag_cols].any(axis=1)
print({"raw_rows": raw_rows, "working_rows": len(df), "event_eligible": int(event_mask.sum()),
       "animal_eligible": int(animal_mask.sum()), "review_rows": int(review_mask.sum())})

df.loc[review_mask].head(10000).to_csv(OUT / "review_sample.csv", index=False, encoding="utf-8-sig")

EXPORT_FULL = False
if EXPORT_FULL:
    df.to_csv(OUT / "milking_flagged.csv", index=False, chunksize=100000)


## 5. Optional model preprocessing
- Descriptive analysis: report missingness and denominators; scaling and blanket imputation are unnecessary.
- Generalization to new animals: split by AnimalId. Missing IDs cannot be reliably grouped.
- Future prediction: use chronological splits. Shift historical features before rolling calculations.
- Fit imputers, scalers, encoders and outlier thresholds on training data only.
- Missing reproduction status can be encoded as Unknown; the modal status is not verified ground truth.
- Never impute AnimalId. Numeric imputation depends on the research question.
- Before-milking prediction of YieldSession must exclude current-event flow, duration and first-two-minute yield because they are unavailable at prediction time.
- Two-minute prediction may use early measurements, but not whole-session average flow or duration.
The optional example below preprocesses a conservative complete-identity cohort for prediction on unseen animals; it does not train or evaluate a model. Install scikit-learn if enabling it.

In [ ]:
RUN_MODEL_PREPROCESSING = False
if RUN_MODEL_PREPROCESSING:
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    numeric_features = ["LactationNumber", "DaysInMilk"]
    categorical_features = ["ReproductionStatus", "milking"]
    features = numeric_features + categorical_features
    eligible = animal_mask & df["YieldSession"].notna() & ~df["flag_key_conflict"] & ~df["flag_exact_duplicate"]
    model_data = df.loc[eligible, features + ["AnimalId", "YieldSession"]].copy()
    for c in categorical_features:
        model_data[c] = model_data[c].astype("string").fillna("Unknown").astype(str)
    split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(split.split(model_data, groups=model_data["AnimalId"]))
    train = model_data.iloc[train_idx]
    test = model_data.iloc[test_idx]
    assert set(train["AnimalId"]).isdisjoint(set(test["AnimalId"]))
    preprocess = ColumnTransformer([
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                              ("scale", StandardScaler())]), numeric_features),
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
    X_train = preprocess.fit_transform(train[features])
    X_test = preprocess.transform(test[features])
    y_train, y_test = train["YieldSession"], test["YieldSession"]
    print(X_train.shape, X_test.shape)


References:
- https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- https://scikit-learn.org/stable/common_pitfalls.html
The original notebook runs sweetviz and contains an unfinished ProfileReport cell with invalid indentation. This notebook provides a separate executable workflow.